In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("1/5: Loading datasets...")
train = pd.read_csv('train.csv')
public_test = pd.read_csv('public_test.csv')
private_test = pd.read_csv('private_test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

for df in [train, public_test, private_test, sample_sub]:
    df.columns = df.columns.str.replace(' ', '_').str.strip()

full_train = pd.concat([train, public_test], ignore_index=True)
X_train = full_train.drop(columns=['User_ID', 'Converted'], errors='ignore')
y_train = full_train['Converted'].astype(int)
X_test = private_test.drop(columns=['User_ID'], errors='ignore')
print("Successfully loaded!")

In [ ]:
print("2/5: Creating features...")
categorical_cols = ['Device_Type', 'Traffic_Source', 'City_Tier', 'Campaign_Code']
categorical_cols = [c for c in categorical_cols if c in X_train.columns]

def engineer_features(df):
    df = df.copy()
    pages = df['Pages_Viewed'] if 'Pages_Viewed' in df.columns else 0
    products = df['Products_Viewed'] if 'Products_Viewed' in df.columns else 0
    time_site = df['Time_On_Site'] if 'Time_On_Site' in df.columns else 0
    age = df['Age'] if 'Age' in df.columns else 30
    income = df['Income'] if 'Income' in df.columns else 50000

    df['Pages_per_Product'] = pages / (products + 1e-5)
    df['Time_per_Page'] = time_site / (pages + 1e-5)
    df['Total_Engagement_Score'] = (pages * 0.4) + (time_site * 0.6)
    df['Income_to_Age_Ratio'] = income / (age + 1e-5)
    return df

X_train_eng = engineer_features(X_train)
X_test_eng = engineer_features(X_test)

for col in categorical_cols:
    X_train_eng[col] = X_train_eng[col].astype(str)
    X_test_eng[col] = X_test_eng[col].astype(str)
    le = LabelEncoder()
    combined_series = pd.concat([X_train_eng[col], X_test_eng[col]], axis=0)
    le.fit(combined_series)
    X_train_eng[col] = le.transform(X_train_eng[col])
    X_test_eng[col] = le.transform(X_test_eng[col])
print("Successfully engineered!")

In [ ]:
print("3/5: Training model layers...")
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_eng))
test_preds = np.zeros(len(X_test_eng))

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbose': -1,
    'random_state': 42,
    'n_estimators': 1500
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_eng, y_train)):
    X_tr, y_tr = X_train_eng.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train_eng.iloc[val_idx], y_train.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test_eng)[:, 1] / n_splits
    print(f"-> Fold {fold + 1} processing complete.")

In [ ]:
print("4/5 & 5/5: Calibrating metric accuracy and exporting submission...")
best_threshold = 0.5
best_f1 = 0
for th in np.arange(0.1, 0.9, 0.01):
    score = f1_score(y_train, (oof_preds >= th).astype(int), average='binary')
    if score > best_f1:
        best_f1 = score
        best_threshold = th

final_predictions = (test_preds >= best_threshold).astype(int)
orig_id_col = 'User ID' if 'User ID' in private_test.columns else 'User_ID'

submission = pd.DataFrame({
    orig_id_col: private_test[orig_id_col],
    'Converted': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("\n[COMPLETE SUCCESS] submission.csv has been built perfectly!")